In [1]:
import re
import glob
import xlwings as xw
import math
from pathlib import Path
import random
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as mpl
from matplotlib import cm
import matplotlib.pyplot as plt
import matplotlib.colors as mclr
import plotly.express as px
import statsmodels.api as sm
import pylab as py
import os, cv2, glob, tempfile
import joblib

from scipy import stats
from scipy.stats import pearsonr

import sklearn
from sklearn import datasets
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import minmax_scale
from sklearn.preprocessing import MaxAbsScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import QuantileTransformer
from sklearn.datasets import make_blobs

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import VotingClassifier
from sklearn import metrics
from sklearn.metrics import recall_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
from sklearn.metrics import matthews_corrcoef
make_scorer = sklearn.metrics.make_scorer
f1 = make_scorer(f1_score, pos_label=1, average="binary")
from sklearn.metrics import classification_report
from sklearn.inspection import permutation_importance
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, KFold
from sklearn import linear_model
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.model_selection import RepeatedStratifiedKFold

import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import BatchNormalization, Conv2D, MaxPooling2D, Activation, Dropout, Flatten, Dense
from keras.optimizers import Adam
from keras import utils as np_utils
from keras import models
import keras_tuner as kt
from scipy import signal

import config as config
print(config.__path__)
from keras.callbacks import EarlyStopping
from keras.utils import to_categorical
from keras import backend as K

import argparse
# from sklearn.utils import class_weight
print(tf.__version__)
print(keras.__version__)

import shutil
import xlwings as xw

%matplotlib inline
mpl.use("Agg")

['C:\\Users\\user\\anaconda3\\lib\\site-packages\\config']
2.12.0-dev20221107
2.12.0


In [2]:
import warnings
warnings.filterwarnings("ignore")

# 1. Data Preparation

In [22]:
## input DirectInfusion set ## 6711 x 16 df
df_ROI_pm = pd.read_csv(r"I:\4_output_FIBproj\4_3_Output_raMSIn\modeling\df_PMDIn_raMSIn_norm_123.csv").reset_index().drop(columns=["index", "Unnamed: 0"])

In [23]:
df_ROI_pm

,pixel_id,250.1448,303.2329,295.2278,480.3099,215.0326,255.233,327.233,619.2894,865.5025,311.1688,269.2486,738.5062,435.2964,514.2848,type,predicted012,noRisk,Risk,Cancer
0,HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_M...,-0.36933,0.088734,0.452591,0.624357,0.744042,0.345283,0.982864,0.291689,1.040174,0.290187,-0.094526,0.321747,0.085217,-0.445423,0,1,0,1,0
1,HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_M...,-0.36933,-0.201833,0.604872,0.460051,-0.810637,0.452955,0.308910,0.058328,0.686840,0.182935,0.328750,0.392576,-0.114069,-0.484110,0,1,0,1,0
2,HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_M...,-0.36933,-0.164687,0.249824,0.892365,0.782306,0.048529,0.107290,-0.656984,0.663825,-0.113763,0.823456,0.112642,-1.114433,-0.465705,0,0,1,0,0
3,HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_M...,-0.36933,-0.097753,0.731184,0.541603,-0.810637,0.430904,0.704984,-0.434250,1.963588,-0.037164,0.561729,1.260449,0.465858,-0.494882,0,0,1,0,0
4,HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_M...,-0.36933,-0.602424,0.280637,-0.087438,0.419113,-0.318601,-0.163125,-0.861845,-0.297223,-0.306744,-0.131589,-0.793509,-0.441613,-0.510262,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3359,HKBULiver_Halfyear_FibrosisLiver_DirectIn_Mice...,-0.36933,0.200524,-0.647980,0.233060,-0.810637,0.765507,0.191102,-0.075249,-0.218766,1.475244,0.340192,-0.500204,0.509061,-0.627430,1,1,0,1,0
3360,HKBULiver_Halfyear_FibrosisLiver_DirectIn_Mice...,-0.36933,0.344717,-0.208700,0.423365,-0.810637,1.252085,0.132593,0.278179,-0.036554,1.534465,1.317275,-0.826349,-0.141638,-0.623744,1,1,0,1,0
3361,HKBULiver_Halfyear_FibrosisLiver_DirectIn_Mice...,-0.36933,0.349257,-0.228705,0.880851,0.195907,1.383750,0.084254,0.416100,0.537164,1.555862,0.725883,0.813694,0.337516,-0.652096,1,1,0,1,0
3362,HKBULiver_Halfyear_FibrosisLiver_DirectIn_Mice...,-0.36933,-0.221511,-0.812250,-0.337346,-0.810637,0.318545,-0.159116,0.150794,-0.189410,1.157331,-0.444499,-0.251843,-0.059958,-0.658126,1,1,0,1,0


In [24]:
df_ROI_pm["Sample"] = 0
df_ROI_pm["Row"] = 0
df_ROI_pm["Scan"] = 0
df_ROI_pm_Sample = []
df_ROI_pm_Row = []
df_ROI_pm_Scan = []
for i in range(len(df_ROI_pm["pixel_id"])):
    df_ROI_pm_Sample.append("_".join(str(df_ROI_pm["pixel_id"][i]).split("_")[:-2]))
    df_ROI_pm_Row.append(int(str(df_ROI_pm["pixel_id"][i]).split("_")[-1]))
    df_ROI_pm_Scan.append(int(str(df_ROI_pm["pixel_id"][i]).split("_")[-2]))
df_ROI_pm["Sample"] = df_ROI_pm_Sample
df_ROI_pm["Row"] = df_ROI_pm_Row
df_ROI_pm["Scan"] = df_ROI_pm_Scan
pop_column = df_ROI_pm.pop('Scan')
df_ROI_pm.insert(1, 'Scan', pop_column)
pop_column = df_ROI_pm.pop('Row')
df_ROI_pm.insert(1, 'Row', pop_column)
pop_column = df_ROI_pm.pop('Sample')
df_ROI_pm.insert(1, 'Sample', pop_column)

In [25]:
df_ROI_pm = df_ROI_pm.sort_values(["Sample", "Scan"], ascending = True).reset_index().drop(columns=["index"])

In [26]:
df_ROI_pm

,pixel_id,Sample,Row,Scan,250.1448,303.2329,295.2278,480.3099,215.0326,255.233,...,311.1688,269.2486,738.5062,435.2964,514.2848,type,predicted012,noRisk,Risk,Cancer
0,HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_M...,HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_M...,1,1,-0.36933,0.088734,0.452591,0.624357,0.744042,0.345283,...,0.290187,-0.094526,0.321747,0.085217,-0.445423,0,1,0,1,0
1,HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_M...,HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_M...,1,2,-0.36933,-0.201833,0.604872,0.460051,-0.810637,0.452955,...,0.182935,0.328750,0.392576,-0.114069,-0.484110,0,1,0,1,0
2,HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_M...,HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_M...,1,3,-0.36933,-0.164687,0.249824,0.892365,0.782306,0.048529,...,-0.113763,0.823456,0.112642,-1.114433,-0.465705,0,0,1,0,0
3,HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_M...,HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_M...,1,4,-0.36933,-0.097753,0.731184,0.541603,-0.810637,0.430904,...,-0.037164,0.561729,1.260449,0.465858,-0.494882,0,0,1,0,0
4,HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_M...,HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_M...,1,5,-0.36933,-0.602424,0.280637,-0.087438,0.419113,-0.318601,...,-0.306744,-0.131589,-0.793509,-0.441613,-0.510262,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3359,HKBULiver_Halfyear_FibrosisLiver_DirectIn_Mice...,HKBULiver_Halfyear_FibrosisLiver_DirectIn_Mice...,1,200,-0.36933,0.200524,-0.647980,0.233060,-0.810637,0.765507,...,1.475244,0.340192,-0.500204,0.509061,-0.627430,1,1,0,1,0
3360,HKBULiver_Halfyear_FibrosisLiver_DirectIn_Mice...,HKBULiver_Halfyear_FibrosisLiver_DirectIn_Mice...,1,201,-0.36933,0.344717,-0.208700,0.423365,-0.810637,1.252085,...,1.534465,1.317275,-0.826349,-0.141638,-0.623744,1,1,0,1,0
3361,HKBULiver_Halfyear_FibrosisLiver_DirectIn_Mice...,HKBULiver_Halfyear_FibrosisLiver_DirectIn_Mice...,1,202,-0.36933,0.349257,-0.228705,0.880851,0.195907,1.383750,...,1.555862,0.725883,0.813694,0.337516,-0.652096,1,1,0,1,0
3362,HKBULiver_Halfyear_FibrosisLiver_DirectIn_Mice...,HKBULiver_Halfyear_FibrosisLiver_DirectIn_Mice...,1,203,-0.36933,-0.221511,-0.812250,-0.337346,-0.810637,0.318545,...,1.157331,-0.444499,-0.251843,-0.059958,-0.658126,1,1,0,1,0


In [27]:
HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M182 = df_ROI_pm.loc[df_ROI_pm["Sample"] == "HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M182"].sort_values(["Row", "Scan"], ascending = [True, True]).reset_index().drop(columns=["index"])
HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M185 = df_ROI_pm.loc[df_ROI_pm["Sample"] == "HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M185"].sort_values(["Row", "Scan"], ascending = [True, True]).reset_index().drop(columns=["index"])
HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M187 = df_ROI_pm.loc[df_ROI_pm["Sample"] == "HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M187"].sort_values(["Row", "Scan"], ascending = [True, True]).reset_index().drop(columns=["index"])
HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M323 = df_ROI_pm.loc[df_ROI_pm["Sample"] == "HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M323"].sort_values(["Row", "Scan"], ascending = [True, True]).reset_index().drop(columns=["index"])
HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M324 = df_ROI_pm.loc[df_ROI_pm["Sample"] == "HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M324"].sort_values(["Row", "Scan"], ascending = [True, True]).reset_index().drop(columns=["index"])
HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M325 = df_ROI_pm.loc[df_ROI_pm["Sample"] == "HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M325"].sort_values(["Row", "Scan"], ascending = [True, True]).reset_index().drop(columns=["index"])
HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M327 = df_ROI_pm.loc[df_ROI_pm["Sample"] == "HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M327"].sort_values(["Row", "Scan"], ascending = [True, True]).reset_index().drop(columns=["index"])
HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M397 = df_ROI_pm.loc[df_ROI_pm["Sample"] == "HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M397"].sort_values(["Row", "Scan"], ascending = [True, True]).reset_index().drop(columns=["index"])
HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M201 = df_ROI_pm.loc[df_ROI_pm["Sample"] == "HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M201"].sort_values(["Row", "Scan"], ascending = [True, True]).reset_index().drop(columns=["index"])
HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M207 = df_ROI_pm.loc[df_ROI_pm["Sample"] == "HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M207"].sort_values(["Row", "Scan"], ascending = [True, True]).reset_index().drop(columns=["index"])
HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M210 = df_ROI_pm.loc[df_ROI_pm["Sample"] == "HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M210"].sort_values(["Row", "Scan"], ascending = [True, True]).reset_index().drop(columns=["index"])
HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M211 = df_ROI_pm.loc[df_ROI_pm["Sample"] == "HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M211"].sort_values(["Row", "Scan"], ascending = [True, True]).reset_index().drop(columns=["index"])
HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M347 = df_ROI_pm.loc[df_ROI_pm["Sample"] == "HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M347"].sort_values(["Row", "Scan"], ascending = [True, True]).reset_index().drop(columns=["index"])
HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M350 = df_ROI_pm.loc[df_ROI_pm["Sample"] == "HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M350"].sort_values(["Row", "Scan"], ascending = [True, True]).reset_index().drop(columns=["index"])
HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M351 = df_ROI_pm.loc[df_ROI_pm["Sample"] == "HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M351"].sort_values(["Row", "Scan"], ascending = [True, True]).reset_index().drop(columns=["index"])
HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_MN4 = df_ROI_pm.loc[df_ROI_pm["Sample"] == "HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_MN4"].sort_values(["Row", "Scan"], ascending = [True, True]).reset_index().drop(columns=["index"])

In [28]:
sample_ROI_list = [HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M182, 
                   HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M185, 
                   HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M187, 
                   HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M323, 
                   HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M324, 
                   HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M325, 
                   HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M327, 
                   HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M397, 
                   HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M201, 
                   HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M207, 
                   HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M210, 
                   HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M211, 
                   HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M347, 
                   HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M350, 
                   HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M351, 
                   HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_MN4]

In [29]:
name_list = ["HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M182", 
                   "HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M185", 
                   "HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M187", 
                   "HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M323", 
                   "HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M324", 
                   "HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M325", 
                   "HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M327", 
                   "HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M397", 
                   "HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M201", 
                   "HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M207", 
                   "HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M210", 
                   "HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M211", 
                   "HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M347", 
                   "HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M350", 
                   "HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M351", 
                   "HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_MN4"]

# 2. Prediction (made)

## PM2.5 Direct Infusion

In [36]:
( list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M182["predicted012"]).count(2) / (list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M182["predicted012"]).count(0) + list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M182["predicted012"]).count(1) + list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M182["predicted012"]).count(2)) ) *100, "%"

(0.0, '%')

In [31]:
# 0: 91.3%, 1: 8.7%, 2: 0.0%

In [88]:
( list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M185["predicted012"]).count(2) / (list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M185["predicted012"]).count(0) + list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M185["predicted012"]).count(1) + list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M185["predicted012"]).count(2)) ) *100, "%"

(0.0, '%')

In [ ]:
# 0: 67.6%, 1: 32.4%, 2: 0.0%

In [42]:
( list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M187["predicted012"]).count(2) / (list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M187["predicted012"]).count(0) + list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M187["predicted012"]).count(1) + list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M187["predicted012"]).count(2)) ) *100, "%"

(6.222222222222222, '%')

In [ ]:
# 0: 40.9%, 1: 52.9%, 2: 6.2%

In [45]:
( list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M323["predicted012"]).count(2) / (list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M323["predicted012"]).count(0) + list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M323["predicted012"]).count(1) + list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M323["predicted012"]).count(2)) ) *100, "%"

(5.263157894736842, '%')

In [34]:
# 0: 68.9%, 1: 25.8%, 2: 5.3%

In [48]:
( list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M324["predicted012"]).count(2) / (list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M324["predicted012"]).count(0) + list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M324["predicted012"]).count(1) + list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M324["predicted012"]).count(2)) ) *100, "%"

(0.0, '%')

In [46]:
# 0: 25.9%, 1: 74.1%, 2: 0.0%

In [51]:
( list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M325["predicted012"]).count(2) / (list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M325["predicted012"]).count(0) + list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M325["predicted012"]).count(1) + list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M325["predicted012"]).count(2)) ) *100, "%"

(1.4423076923076923, '%')

In [45]:
# 0: 52.9%, 1: 45.7%, 2: 1.4%

In [54]:
( list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M327["predicted012"]).count(2) / (list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M327["predicted012"]).count(0) + list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M327["predicted012"]).count(1) + list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M327["predicted012"]).count(2)) ) *100, "%"

(0.0, '%')

In [44]:
# 0: 4.4%, 1: 95.6%, 2: 0.0%

In [57]:
( list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M397["predicted012"]).count(2) / (list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M397["predicted012"]).count(0) + list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M397["predicted012"]).count(1) + list(HKBULiver_Halfyear_FibrosisAMCLiver_DirectIn_MiceNL_M397["predicted012"]).count(2)) ) *100, "%"

(0.0, '%')

In [ ]:
# 0: 71.7%, 1: 28.3%, 2: 0.0%

In [60]:
( list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M201["predicted012"]).count(2) / (list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M201["predicted012"]).count(0) + list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M201["predicted012"]).count(1) + list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M201["predicted012"]).count(2)) ) *100, "%"

(0.0, '%')

In [ ]:
# 0: 33.0%, 1: 67.0%, 2: 0.0%

In [91]:
( list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M207["predicted012"]).count(2) / (list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M207["predicted012"]).count(0) + list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M207["predicted012"]).count(1) + list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M207["predicted012"]).count(2)) ) *100, "%"

(1.4492753623188406, '%')

In [ ]:
# 0: 65.2%, 1: 33.3%, 2: 1.4%

In [66]:
( list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M210["predicted012"]).count(2) / (list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M210["predicted012"]).count(0) + list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M210["predicted012"]).count(1) + list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M210["predicted012"]).count(2)) ) *100, "%"

(5.288461538461538, '%')

In [ ]:
# 0: 13.9%, 1: 80.8%, 2: 5.3%

In [69]:
( list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M211["predicted012"]).count(2) / (list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M211["predicted012"]).count(0) + list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M211["predicted012"]).count(1) + list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M211["predicted012"]).count(2)) ) *100, "%"

(0.0, '%')

In [ ]:
# 0: 6.9%, 1: 93.1%, 2: 0.0%

In [72]:
( list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M347["predicted012"]).count(2) / (list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M347["predicted012"]).count(0) + list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M347["predicted012"]).count(1) + list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M347["predicted012"]).count(2)) ) *100, "%"

(0.0, '%')

In [ ]:
# 0: 23.5%, 1: 76.5%, 2: 0.0%

In [75]:
( list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M350["predicted012"]).count(2) / (list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M350["predicted012"]).count(0) + list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M350["predicted012"]).count(1) + list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M350["predicted012"]).count(2)) ) *100, "%"

(0.0, '%')

In [ ]:
# 0: 29.0%, 1: 71.0%, 2: 0.0%

In [78]:
( list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M351["predicted012"]).count(2) / (list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M351["predicted012"]).count(0) + list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M351["predicted012"]).count(1) + list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_M351["predicted012"]).count(2)) ) *100, "%"

(7.171314741035857, '%')

In [ ]:
# 0: 1.2%, 1: 91.6%, 2: 7.2%

In [81]:
( list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_MN4["predicted012"]).count(2) / (list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_MN4["predicted012"]).count(0) + list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_MN4["predicted012"]).count(1) + list(HKBULiver_Halfyear_FibrosisLiver_DirectIn_MiceFIB_MN4["predicted012"]).count(2)) ) *100, "%"

(0.0, '%')

In [ ]:
# 0: 2.5%, 1: 97.5%, 2: 0.0%